In [ ]:
import os

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

from pathlib import Path
from contextlib import nullcontext
import gc
import json
import random
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
from torch import nn
from transformers import AutoModelForSeq2SeqLM, AutoModelForSequenceClassification

SEED = 13
PRECISION = "bf16-mixed"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEQ_LEN = 256
MAX_QUERY_LEN = 32
MAX_DOC_LEN = 221

LATENCY_BATCH_SIZE = 1
LATENCY_WARMUP_STEPS = 10
LATENCY_MEASURE_STEPS = 50

# Path to the project directory that contains the local model folders.
MODEL_ROOT = Path("..") / "models"
# Directory for generated latency JSON files.
OUTPUT_DIR = Path.cwd()

MODELS_TO_BENCHMARK = (
    "monot5_base",
    "bge_reranker",
)

MODEL_SPECS = {
    "monot5_base": {
        "label": "monot5-base-msmarco",
        "kind": "monot5_seq2seq",
        "path": MODEL_ROOT / "monot5-base-msmarco",
    },
    "bge_reranker": {
        "label": "bge-reranker-v2-m3",
        "kind": "sequence_classification",
        "path": MODEL_ROOT / "bge-reranker-v2-m3",
    },
}

MODEL_LOAD_DTYPE = None

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def resolve_precision(requested_precision: str, device_kind: str) -> str:
    if requested_precision == "bf16-mixed":
        if device_kind == "cuda" and torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            return requested_precision
        print(f'Falling back from precision={requested_precision!r} to "32-true" because bf16 is not available.')
        return "32-true"
    if requested_precision == "16-mixed" and device_kind != "cuda":
        print(f'Falling back from precision={requested_precision!r} to "32-true" because fp16 autocast needs CUDA.')
        return "32-true"
    return requested_precision

def setup_device(requested_device: str) -> torch.device:
    if requested_device.startswith("cuda") and not torch.cuda.is_available():
        return torch.device("cpu")
    return torch.device(requested_device)

def get_autocast_context(device: torch.device, precision: str):
    if device.type != "cuda":
        return nullcontext()
    if precision == "bf16-mixed":
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    if precision == "16-mixed":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()

def model_device(model: nn.Module) -> torch.device:
    return next(model.parameters()).device

def model_autocast_context(model: nn.Module):
    return get_autocast_context(model_device(model), EFFECTIVE_PRECISION)

def count_model_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())

def count_trainable_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

seed_everything(SEED)
device = setup_device(DEVICE)
EFFECTIVE_PRECISION = resolve_precision(PRECISION, device.type)

print(json.dumps({
    "models_to_benchmark": MODELS_TO_BENCHMARK,
    "device": str(device),
    "precision": EFFECTIVE_PRECISION,
    "seq_len": SEQ_LEN,
    "max_query_len": MAX_QUERY_LEN,
    "max_doc_len": MAX_DOC_LEN,
    "latency_batch_size": LATENCY_BATCH_SIZE,
    "latency_warmup_steps": LATENCY_WARMUP_STEPS,
    "latency_measure_steps": LATENCY_MEASURE_STEPS,
    "output_dir": str(OUTPUT_DIR),
    "offline_flags": {
        "TRANSFORMERS_OFFLINE": os.environ.get("TRANSFORMERS_OFFLINE"),
        "HF_HUB_OFFLINE": os.environ.get("HF_HUB_OFFLINE"),
        "HF_DATASETS_OFFLINE": os.environ.get("HF_DATASETS_OFFLINE"),
    },
}, indent=2))


In [ ]:
def get_config_int(config, name: str, default: Optional[int] = None) -> Optional[int]:
    value = getattr(config, name, default)
    return int(value) if value is not None else None

def make_random_input_ids(
    batch_size: int,
    seq_len: int,
    vocab_size: int,
    device: torch.device,
    low_token_id: int = 3,
) -> torch.Tensor:
    high = max(int(vocab_size), low_token_id + 1)
    return torch.randint(low_token_id, high, (batch_size, seq_len), dtype=torch.long, device=device)

def make_attention_mask(batch_size: int, seq_len: int, device: torch.device) -> torch.Tensor:
    return torch.ones((batch_size, seq_len), dtype=torch.long, device=device)

def prepare_monot5_batch(model: nn.Module, batch_size: int, seq_len: int, device: torch.device) -> Tuple[Dict[str, torch.Tensor], Dict[str, object]]:
    config = model.config
    vocab_size = get_config_int(config, "vocab_size")
    if vocab_size is None:
        raise ValueError("MonoT5 config does not expose vocab_size")

    input_ids = make_random_input_ids(batch_size, seq_len, vocab_size, device)
    eos_token_id = get_config_int(config, "eos_token_id")
    if eos_token_id is not None:
        input_ids[:, -1] = eos_token_id

    decoder_start_token_id = get_config_int(config, "decoder_start_token_id")
    if decoder_start_token_id is None:
        decoder_start_token_id = get_config_int(config, "pad_token_id")
    if decoder_start_token_id is None:
        raise ValueError("Could not determine decoder_start_token_id or pad_token_id for MonoT5 forward pass.")

    batch = {
        "input_ids": input_ids,
        "attention_mask": make_attention_mask(batch_size, seq_len, device),
        "decoder_input_ids": torch.full(
            (batch_size, 1),
            decoder_start_token_id,
            dtype=torch.long,
            device=device,
        ),
    }
    metadata = {
        "synthetic_batch": True,
        "tokenizer_timed": False,
        "input_ids_shape": list(input_ids.shape),
        "decoder_input_ids_shape": list(batch["decoder_input_ids"].shape),
        "vocab_size": vocab_size,
        "eos_token_id": eos_token_id,
        "decoder_start_token_id": decoder_start_token_id,
    }
    return batch, metadata

def prepare_sequence_classification_batch(model: nn.Module, batch_size: int, seq_len: int, device: torch.device) -> Tuple[Dict[str, torch.Tensor], Dict[str, object]]:
    config = model.config
    vocab_size = get_config_int(config, "vocab_size")
    if vocab_size is None:
        raise ValueError("Sequence-classification config does not expose vocab_size")

    input_ids = make_random_input_ids(batch_size, seq_len, vocab_size, device)
    bos_token_id = get_config_int(config, "bos_token_id", 0)
    eos_token_id = get_config_int(config, "eos_token_id", 2)

    query_end = min(MAX_QUERY_LEN + 1, seq_len - 1)
    input_ids[:, 0] = bos_token_id if bos_token_id is not None else input_ids[:, 0]
    input_ids[:, query_end] = eos_token_id if eos_token_id is not None else input_ids[:, query_end]
    if query_end + 1 < seq_len:
        input_ids[:, query_end + 1] = eos_token_id if eos_token_id is not None else input_ids[:, query_end + 1]
    input_ids[:, -1] = eos_token_id if eos_token_id is not None else input_ids[:, -1]

    batch = {
        "input_ids": input_ids,
        "attention_mask": make_attention_mask(batch_size, seq_len, device),
    }
    metadata = {
        "synthetic_batch": True,
        "tokenizer_timed": False,
        "input_ids_shape": list(input_ids.shape),
        "vocab_size": vocab_size,
        "bos_token_id": bos_token_id,
        "eos_token_id": eos_token_id,
        "query_end_index": query_end,
    }
    return batch, metadata

def forward_monot5_score(model: nn.Module, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        decoder_input_ids=batch["decoder_input_ids"],
    )
    first_token_logits = outputs.logits[:, 0, :]
    return first_token_logits[:, 0]

def forward_sequence_classification_score(model: nn.Module, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
    outputs = model(**batch)
    return outputs.logits.squeeze(-1)

def synchronize_for_latency(device: torch.device) -> bool:
    if device.type == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize(device)
        return True
    return False


In [ ]:
def load_latency_target(model_key: str, device: torch.device) -> Dict[str, object]:
    if model_key not in MODEL_SPECS:
        raise ValueError(f"Unknown model_key={model_key!r}; choose one of {sorted(MODEL_SPECS)}")

    spec = MODEL_SPECS[model_key]
    model_path = Path(spec["path"])

    load_kwargs = {"local_files_only": True}
    if MODEL_LOAD_DTYPE is not None:
        load_kwargs["torch_dtype"] = MODEL_LOAD_DTYPE

    if spec["kind"] == "monot5_seq2seq":
        model = AutoModelForSeq2SeqLM.from_pretrained(str(model_path), **load_kwargs)
        model.to(device)
        model.eval()
        batch, batch_metadata = prepare_monot5_batch(model, LATENCY_BATCH_SIZE, SEQ_LEN, device)
        forward_fn = lambda: forward_monot5_score(model, batch)
    elif spec["kind"] == "sequence_classification":
        model = AutoModelForSequenceClassification.from_pretrained(str(model_path), **load_kwargs)
        model.to(device)
        model.eval()
        batch, batch_metadata = prepare_sequence_classification_batch(model, LATENCY_BATCH_SIZE, SEQ_LEN, device)
        forward_fn = lambda: forward_sequence_classification_score(model, batch)
    else:
        raise ValueError(f"Unsupported model kind for {model_key}: {spec['kind']!r}")

    return {
        "model_key": model_key,
        "label": spec["label"],
        "kind": spec["kind"],
        "model_path": str(model_path),
        "model": model,
        "batch": batch,
        "batch_metadata": batch_metadata,
        "forward_fn": forward_fn,
    }

def measure_forward_latency(
    target: Dict[str, object],
    warmup_steps: int,
    measure_steps: int,
) -> Dict[str, object]:
    warmup_steps = max(0, int(warmup_steps))
    measure_steps = int(measure_steps)
    if measure_steps <= 0:
        raise ValueError("measure_steps must be positive")

    model = target["model"]
    forward_fn = target["forward_fn"]
    device = model_device(model)
    batch_metadata = target["batch_metadata"]
    batch_size = int(batch_metadata["input_ids_shape"][0])
    seq_len = int(batch_metadata["input_ids_shape"][1])
    elapsed_ms: List[float] = []
    was_training = model.training

    model.eval()
    try:
        with torch.inference_mode(), model_autocast_context(model):
            last_scores = None
            for _ in range(warmup_steps):
                last_scores = forward_fn()

            synchronize_for_latency(device)
            for _ in range(measure_steps):
                synchronize_for_latency(device)
                start = time.perf_counter()
                last_scores = forward_fn()
                synchronize_for_latency(device)
                elapsed_ms.append((time.perf_counter() - start) * 1000.0)
    finally:
        if was_training:
            model.train()

    values = np.asarray(elapsed_ms, dtype=np.float64)
    total_sec = float(values.sum() / 1000.0)
    return {
        "latency_scope": "synthetic_prepared_batch_model_forward_only",
        "tokenizer_timed": False,
        "model_loading_timed": False,
        "generation_timed": False,
        "model_key": target["model_key"],
        "model_label": target["label"],
        "model_kind": target["kind"],
        "model_path": target["model_path"],
        "batch_size": batch_size,
        "seq_len": seq_len,
        "warmup_steps": warmup_steps,
        "measure_steps": measure_steps,
        "precision": EFFECTIVE_PRECISION,
        "device": str(device),
        "cuda_synchronized": bool(device.type == "cuda" and torch.cuda.is_available()),
        "latency_ms_mean": float(values.mean()),
        "latency_ms_std": float(values.std()),
        "latency_ms_min": float(values.min()),
        "latency_ms_p50": float(np.percentile(values, 50)),
        "latency_ms_p95": float(np.percentile(values, 95)),
        "latency_ms_max": float(values.max()),
        "throughput_examples_per_sec": float(batch_size * measure_steps / total_sec) if total_sec > 0 else None,
        "score_shape": list(last_scores.shape) if isinstance(last_scores, torch.Tensor) else None,
    }

def unload_latency_target(target: Optional[Dict[str, object]]) -> None:
    if target is None:
        return
    target.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
all_summaries = []

for model_key in MODELS_TO_BENCHMARK:
    print(f"\n=== Benchmarking {model_key} ===")
    target = None
    try:
        target = load_latency_target(model_key, device)
        model = target["model"]
        latency_result = measure_forward_latency(
            target=target,
            warmup_steps=LATENCY_WARMUP_STEPS,
            measure_steps=LATENCY_MEASURE_STEPS,
        )
        summary = {
            "model_key": model_key,
            "model_label": target["label"],
            "model_kind": target["kind"],
            "model_path": target["model_path"],
            "total_parameters": count_model_parameters(model),
            "trainable_parameters": count_trainable_parameters(model),
            "device": str(device),
            "precision": EFFECTIVE_PRECISION,
            "seq_len": SEQ_LEN,
            "max_query_len": MAX_QUERY_LEN,
            "max_doc_len": MAX_DOC_LEN,
            "latency_batch_size": LATENCY_BATCH_SIZE,
            "prepared_batch": target["batch_metadata"],
            "latency_result": latency_result,
        }
        all_summaries.append(summary)

        print(json.dumps(latency_result, indent=2))
        print(json.dumps({
            "model_key": model_key,
            "total_parameters": summary["total_parameters"],
            "trainable_parameters": summary["trainable_parameters"],
        }, indent=2))

        per_model_path = OUTPUT_DIR / f"{model_key}_forward_latency_batch1.json"
        per_model_path.parent.mkdir(parents=True, exist_ok=True)
        per_model_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
        print(f"Saved latency summary: {per_model_path}")
    finally:
        unload_latency_target(target)

combined_summary = {
    "benchmark_name": "sota_latency",
    "models_to_benchmark": MODELS_TO_BENCHMARK,
    "device": str(device),
    "precision": EFFECTIVE_PRECISION,
    "latency_batch_size": LATENCY_BATCH_SIZE,
    "seq_len": SEQ_LEN,
    "warmup_steps": LATENCY_WARMUP_STEPS,
    "measure_steps": LATENCY_MEASURE_STEPS,
    "results": all_summaries,
}

combined_path = OUTPUT_DIR / "sota_forward_latency_batch1.json"
combined_path.write_text(json.dumps(combined_summary, indent=2), encoding="utf-8")
print(f"\nSaved combined latency summary: {combined_path}")

combined_summary
